In [5]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.model_selection import train_test_split

In [6]:
# ── Load your dataset ──────────────────────────────────────────
adata = sc.read_h5ad("SCP1884_subset_50_normalised_expressions_with_embeddings_cp.h5ad")   # change to your file path

print(f"Total cells    : {adata.shape[0]}")
print(f"Total features : {adata.shape[1]}")
print(f"obsm keys      : {list(adata.obsm.keys())}")
print(f"obs columns    : {adata.obs.columns.tolist()}")

Total cells    : 661676
Total features : 25032
obsm keys      : ['X_scGPT']
obs columns    : ['sample_id', 'tissue', 'compartment', 'matrix_type', 'biosample_id', 'n_genes', 'n_counts', 'Chem', 'Site', 'Type', 'donor_id', 'Layer', 'Celltype', 'sex', 'species', 'species__ontology_label', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label', 'organ', 'organ__ontology_label', 'disease', 'disease__ontology_label']


In [3]:
adata.obs["disease__ontology_label"].unique()

['Crohn's disease', 'normal']
Categories (2, object): ['Crohn's disease', 'normal']

In [7]:
# ── Understand your columns ────────────────────────────────────
print("obs columns:", adata.obs.columns.tolist())
print("obsm keys  :", list(adata.obsm.keys()))
print()

# ── Check donor and disease columns ───────────────────────────
print("Unique donors :", adata.obs["donor_id"].nunique())
print("Disease labels:", adata.obs["disease__ontology_label"].unique().tolist())
print()

# ── Per-donor disease breakdown ───────────────────────────────
donor_disease = (
    adata.obs
    .groupby("donor_id")["disease__ontology_label"]
    .first()
    .reset_index()
)
print("Donor disease breakdown:")
print(donor_disease["disease__ontology_label"].value_counts())
print()
print(donor_disease.head(10))

obs columns: ['sample_id', 'tissue', 'compartment', 'matrix_type', 'biosample_id', 'n_genes', 'n_counts', 'Chem', 'Site', 'Type', 'donor_id', 'Layer', 'Celltype', 'sex', 'species', 'species__ontology_label', 'library_preparation_protocol', 'library_preparation_protocol__ontology_label', 'organ', 'organ__ontology_label', 'disease', 'disease__ontology_label']
obsm keys  : ['X_scGPT']

Unique donors : 50
Disease labels: ["Crohn's disease", 'normal']

Donor disease breakdown:
disease__ontology_label
Crohn's disease    36
normal             14
Name: count, dtype: int64

  donor_id disease__ontology_label
0   102141                  normal
1   104152         Crohn's disease
2   104689         Crohn's disease
3   105598         Crohn's disease
4   107306         Crohn's disease
5   109389         Crohn's disease
6   110204         Crohn's disease
7   114902         Crohn's disease
8   115208         Crohn's disease
9   119540         Crohn's disease


/tmp/ipykernel_55985/1941251949.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("donor_id")["disease__ontology_label"]


### Build patient-level label map

In [8]:
# ── Label map — must match exactly what's in disease column ───
LABEL_MAP = {
    "normal"          : 0,
    "Crohn's disease" : 1
}

EMBEDDING_KEY = "X_scGPT"    # change if needed — check adata.obsm.keys()
RANDOM_SEED   = 42

# ── Build donor → label dictionary ────────────────────────────
donor_info = (
    adata.obs
    .groupby("donor_id")["disease__ontology_label"]
    .first()
    .reset_index()
)
donor_info.columns = ["donor_id", "disease"]

# Map disease string → integer label
donor_info["label"] = donor_info["disease"].map(LABEL_MAP)

# Check for unmapped labels
unmapped = donor_info[donor_info["label"].isna()]
if len(unmapped) > 0:
    print(f"WARNING — unmapped labels: {unmapped['disease'].unique().tolist()}")
    print("Add these to LABEL_MAP before continuing")
else:
    print("All labels mapped successfully ✓")

print("\nDonor label distribution:")
print(donor_info["label"].value_counts())
print(f"\nTotal donors: {len(donor_info)}")

All labels mapped successfully ✓

Donor label distribution:
label
1    36
0    14
Name: count, dtype: int64

Total donors: 50


/tmp/ipykernel_55985/1465240164.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("donor_id")["disease__ontology_label"]


### Stratified train/validation split at donor level

In [9]:
# ── Split at DONOR level — never at cell level ─────────────────
all_donor_ids    = donor_info["donor_id"].tolist()
all_donor_labels = donor_info["label"].tolist()

train_donors, val_donors = train_test_split(
    all_donor_ids,
    test_size    = 0.2,              # 80% discovery, 20% validation
    random_state = RANDOM_SEED,
    stratify     = all_donor_labels  # preserves Crohn/Normal ratio in both splits
)

print(f"Discovery  donors : {len(train_donors)}")
print(f"Validation donors : {len(val_donors)}")

# ── Verify no overlap ──────────────────────────────────────────
overlap = set(train_donors) & set(val_donors)
assert len(overlap) == 0, f"Leakage detected: {overlap}"
print("No donor overlap confirmed ✓")

# ── Subset adata ───────────────────────────────────────────────
adata_discovery  = adata[adata.obs["donor_id"].isin(train_donors)].copy()
adata_validation = adata[adata.obs["donor_id"].isin(val_donors)].copy()

print(f"\nDiscovery  cells : {adata_discovery.shape[0]}")
print(f"Validation cells : {adata_validation.shape[0]}")

Discovery  donors : 40
Validation donors : 10
No donor overlap confirmed ✓



Discovery  cells : 535369
Validation cells : 126307


### Updated extraction function for donor_id + disease column

In [13]:
def extract_patient_npy_from_columns(
    adata,
    output_dir,
    labels_path,
    embedding_key  = "X_scGPT",
    donor_col      = "donor_id",
    disease_col    = "disease__ontology_label",
    label_map      = None,
):
    """
    Extracts per-donor embeddings from AnnData and saves as .npy files.
    Labels come from a separate disease column — not from the donor_id string.

    Args:
        adata         : AnnData object
        output_dir    : folder to save per-donor .npy files
        labels_path   : path to save labels .npy
        embedding_key : key in adata.obsm
        donor_col     : column containing donor IDs
        disease_col   : column containing disease label strings
        label_map     : dict mapping disease string → int
                        e.g. {"normal": 0, "Crohn's disease": 1}

    Output structure:
        output_dir/
            donor001_embeddings.npy    (num_cells, d_h)
            donor002_embeddings.npy    (num_cells, d_h)
            ...
        labels_path → (num_donors,) int array [0, 1, 0, ...]
    """

    os.makedirs(output_dir, exist_ok=True)

    donor_ids   = sorted(adata.obs[donor_col].unique())
    labels      = []
    skipped     = []
    cell_counts = []
    last_Z      = None

    print(f"Processing {len(donor_ids)} donors → {output_dir}")
    print("-" * 60)

    for did in donor_ids:

        # ── Get cells for this donor ───────────────────────────
        donor_mask    = adata.obs[donor_col] == did
        donor_adata   = adata[donor_mask]

        # ── Extract disease label ──────────────────────────────
        # Take most common disease label for this donor
        raw_label = donor_adata.obs[disease_col].mode()[0]

        if label_map is not None:
            if raw_label not in label_map:
                print(f"  SKIP {did} — '{raw_label}' not in label_map")
                skipped.append(did)
                continue
            numeric_label = label_map[raw_label]
        else:
            try:
                numeric_label = int(raw_label)
            except ValueError:
                print(f"  SKIP {did} — cannot convert '{raw_label}' to int")
                skipped.append(did)
                continue

        # ── Extract embedding matrix ───────────────────────────
        if embedding_key in donor_adata.obsm:
            Z = donor_adata.obsm[embedding_key]        # (num_cells, d_h)
        else:
            print(f"  WARNING: '{embedding_key}' not in obsm — falling back to X")
            Z = (donor_adata.X.toarray()
                 if hasattr(donor_adata.X, "toarray")
                 else np.array(donor_adata.X))

        # ── Validate ───────────────────────────────────────────
        assert not np.isnan(Z).any(), f"NaN found in donor {did}"
        assert not np.isinf(Z).any(), f"Inf found in donor {did}"

        # ── Save per-donor .npy ────────────────────────────────
        # Clean donor_id for use as filename
        safe_did = str(did).replace("/", "_").replace(" ", "_")
        fname    = f"{safe_did}_embeddings.npy"
        np.save(os.path.join(output_dir, fname), Z.astype(np.float32))

        labels.append(numeric_label)
        cell_counts.append(Z.shape[0])
        last_Z = Z

        print(f"  {str(did):<35} "
              f"label={numeric_label} ({raw_label:<18}) "
              f"cells={Z.shape[0]:>5}")

    # ── Save labels array ──────────────────────────────────────
    np.save(labels_path, np.array(labels, dtype=np.int64))

    # ── Summary ────────────────────────────────────────────────
    labels_arr = np.array(labels)
    print("-" * 60)
    print(f"Saved       : {len(labels)} donors")
    print(f"Skipped     : {len(skipped)} {skipped if skipped else ''}")
    print(f"Labels path : {labels_path}")
    if cell_counts:
        print(f"Cell range  : {min(cell_counts)} – {max(cell_counts)} cells per donor")
        print(f"Embedding   : dim = {last_Z.shape[1]}")
    print("\nClass balance:")
    for name, cls in sorted(label_map.items(), key=lambda x: x[1]):
        n = (labels_arr == cls).sum()
        print(f"  {name:<20} (label={cls}) : "
              f"{n:>3} donors ({n/len(labels_arr)*100:.1f}%)")

    return labels_arr, cell_counts

In [14]:
LABEL_MAP = {
    "normal"          : 0,
    "Crohn's disease" : 1
}

# ── Discovery ──────────────────────────────────────────────────
print("=" * 60)
print("DISCOVERY")
print("=" * 60)
disc_labels, disc_counts = extract_patient_npy_from_columns(
    adata         = adata_discovery,
    output_dir    = "data/discovery/",
    labels_path   = "data/discovery_labels.npy",
    embedding_key = EMBEDDING_KEY,
    donor_col     = "donor_id",
    disease_col   = "disease__ontology_label",
    label_map     = LABEL_MAP
)

# ── Validation ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)
val_labels, val_counts = extract_patient_npy_from_columns(
    adata         = adata_validation,
    output_dir    = "data/validation/",
    labels_path   = "data/validation_labels.npy",
    embedding_key = EMBEDDING_KEY,
    donor_col     = "donor_id",
    disease_col   = "disease__ontology_label",
    label_map     = LABEL_MAP
)

DISCOVERY
Processing 40 donors → data/discovery/
------------------------------------------------------------


  102141                              label=0 (normal            ) cells=12430


  104152                              label=1 (Crohn's disease   ) cells=15382


  107306                              label=1 (Crohn's disease   ) cells= 9701


  109389                              label=1 (Crohn's disease   ) cells=15245
  110204                              label=1 (Crohn's disease   ) cells= 7702


...

  125985                              label=1 (Crohn's disease   ) cells= 9016
  148519                              label=1 (Crohn's disease   ) cells= 9016


  154787                              label=1 (Crohn's disease   ) cells= 9776
  172428                              label=1 (Crohn's disease   ) cells= 9211


  195045                              label=1 (Crohn's disease   ) cells= 8231


  197396                              label=0 (normal            ) cells=23499
  N13                                 label=0 (normal            ) cells= 5300


  N20                                 label=0 (normal            ) cells= 6563
------------------------------------------------------------
Saved       : 10 donors
Skipped     : 0 
Labels path : data/validation_labels.npy
Cell range  : 5300 – 37637 cells per donor
Embedding   : dim = 512

Class balance:
  normal               (label=0) :   3 donors (30.0%)
  Crohn's disease      (label=1) :   7 donors (70.0%)


In [15]:
# ── File counts ────────────────────────────────────────────────
disc_files = sorted([f for f in os.listdir("data/discovery/")
                     if f.endswith(".npy")])
val_files  = sorted([f for f in os.listdir("data/validation/")
                     if f.endswith(".npy")])

disc_labels_loaded = np.load("data/discovery_labels.npy")
val_labels_loaded  = np.load("data/validation_labels.npy")

print("FINAL CHECKS")
print("-" * 50)
print(f"Discovery  files   : {len(disc_files)}")
print(f"Discovery  labels  : {len(disc_labels_loaded)}")
assert len(disc_files) == len(disc_labels_loaded), "Mismatch in discovery!"
print("Discovery match    : ✓")

print(f"Validation files   : {len(val_files)}")
print(f"Validation labels  : {len(val_labels_loaded)}")
assert len(val_files) == len(val_labels_loaded), "Mismatch in validation!"
print("Validation match   : ✓")

# ── No donor overlap ───────────────────────────────────────────
disc_ids = set([f.replace("_embeddings.npy", "") for f in disc_files])
val_ids  = set([f.replace("_embeddings.npy", "") for f in val_files])
overlap  = disc_ids & val_ids
assert len(overlap) == 0, f"Leakage: {overlap}"
print(f"Donor overlap      : 0 ✓")

# ── Sample file shape ──────────────────────────────────────────
sample      = np.load(f"data/discovery/{disc_files[0]}")
print(f"\nSample file        : {disc_files[0]}")
print(f"Shape              : {sample.shape}   (num_cells, d_h)")
print(f"dtype              : {sample.dtype}")
print(f"NaN                : {np.isnan(sample).any()}")
print(f"Inf                : {np.isinf(sample).any()}")

# ── Expected class balance ─────────────────────────────────────
print(f"\nDiscovery  — Normal: {(disc_labels_loaded==0).sum()}  "
      f"Crohn: {(disc_labels_loaded==1).sum()}")
print(f"Validation — Normal: {(val_labels_loaded==0).sum()}  "
      f"Crohn: {(val_labels_loaded==1).sum()}")

print("\nAll checks passed ✓ — ready for training")

FINAL CHECKS
--------------------------------------------------
Discovery  files   : 40
Discovery  labels  : 40
Discovery match    : ✓
Validation files   : 10
Validation labels  : 10
Validation match   : ✓
Donor overlap      : 0 ✓

Sample file        : 102141_embeddings.npy
Shape              : (12430, 512)   (num_cells, d_h)
dtype              : float32
NaN                : False
Inf                : False

Discovery  — Normal: 11  Crohn: 29
Validation — Normal: 3  Crohn: 7

All checks passed ✓ — ready for training
